# 统一过滤后评分与发布

使用过滤后的真实训练集重新拟合 A3 的 ANOVA 和 PCA。沿用 A1、A2 和 A3 原有评估入口与定义。主实验每个方法评分后更新 README，结果保存在 outputs/min10。

In [ ]:
from pathlib import Path
import gc, json, os, shutil, sys
import anndata as ad
import nbformat
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
from data.baseline import dense32
from data.evaluation import (dense_block, imputed_features, natural_reference_statistics,
                            read_labels, region_pairs, select_features)
from data.paths import OUTPUTS, PROJECT, check_output
from evaluate.A1.metrics import a1_metrics
from evaluate.A1.plotting import load_plot_regions, plot_a1
from evaluate.A2.metrics import paired_pearson
from evaluate.A3.metrics import milisi

METHODS=['simscDNAm','FittedMarginal','simATAC','simCAS','PeakVI','DiTSim']
SUBSETS={'liu2023':['GSE213221','GSE215213','GSE215342'],
         'liu2021':['GSE130553','GSE131354','GSE131360'],
         'tian2023':['GSE167119','GSE167577','GSE167978']}
CASES=['liu2023/full']+[f'{study}/{gse}' for study,accessions in SUBSETS.items() for gse in accessions]
COLUMNS=['A1 1-WD↓','A1 MSE↓','A1 Pearson↑','A2 Pearson↑','A3 miLISI↑']
ROOT=check_output(OUTPUTS/'min10')
RUN=ROOT/'internal/jobs'
FIGURE_PREFIX='/files/data/xb/outputs/simscDNAm/min10/figures/'

def read_csv(path):
    return pd.read_csv(path,float_precision='round_trip')

def write_csv(frame,path):
    path=check_output(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_suffix('.csv.partial')
    frame.to_csv(temporary,index=False,encoding='utf-8-sig')
    temporary.replace(path)

def notebook_function(name, function):
    scope={}
    for cell in nbformat.read(PROJECT/'notebooks'/name,as_version=4).cells[:-1]:
        if cell.cell_type=='code':exec(cell.source,scope)
    return scope[function]

def score(case, method):
    assert case in CASES and method in METHODS
    destination=RUN/case/method
    destination.mkdir(parents=True,exist_ok=True)
    if (destination/'scores.csv').exists():
        return
    prepared=ROOT/'preprocessing'/case
    candidate_path = (
        ROOT/'model'/case/'simscDNAm/data.h5ad' if method == 'simscDNAm'
        else ROOT/'baseline'/case/method/'data.h5ad'
    )
    with threadpool_limits(limits=8):
        real=ad.read_h5ad(prepared/'HeldoutReal.h5ad')
        candidate=ad.read_h5ad(candidate_path)
        assert candidate.shape==real.shape and candidate.var_names.equals(real.var_names)
        assert candidate.obs_names.is_unique
        if sparse.issparse(candidate.X):
            # 逐区间评分频繁切列，先转成同一 dtype 的稠密内存矩阵。
            candidate.X = candidate.X.toarray()
        pd.testing.assert_series_equal(candidate.obs['Subclass'].astype(str).value_counts().sort_index(),
            real.obs['Subclass'].astype(str).value_counts().sort_index(),check_names=False)
        if method in METHODS[4:]:
            for start in range(0,candidate.n_obs,32):
                block=dense32(candidate.X[start:start+32])
                assert np.isfinite(block).all() and block.min()>=0 and block.max()<=1
        print('核验并计算 A1',case,method,flush=True)
        natural=real.n_vars-1000
        real_rows=np.arange(real.n_obs)
        rows=np.arange(candidate.n_obs)
        a1,details=a1_metrics(real,candidate,real_rows,rows,natural)
        write_csv(details,destination/'A1_type_details.csv')
        pairs=region_pairs(real,pair_path=prepared/'region_pairs.csv')
        a2=pd.concat([pairs,paired_pearson(dense_block(candidate,rows,pairs['原region索引'].to_numpy()),
             dense_block(candidate,rows,pairs['复制region索引'].to_numpy()))],axis=1)
        a2.insert(0,'方法',method)
        write_csv(a2,destination/'A2_duplicate_correlations.csv')
        print('计算 A3',case,method,flush=True)
        cached=RUN/case/'pca.npz'
        if not cached.exists():
            train=ad.read_h5ad(prepared/'TrainingReal.h5ad')
            assert not len(train.obs_names.intersection(real.obs_names))
            train_rows = np.arange(train.n_obs)
            fill, scores = natural_reference_statistics(train, train_rows, natural)
            selected = select_features(scores, 50000)
            pca = PCA(n_components=min(50, train.n_obs-1, len(selected)), svd_solver='randomized', random_state=42)
            pca.fit(imputed_features(train, train_rows, selected, fill))
            reference = ROOT/'internal/evaluation'/case
            reference.mkdir(parents=True, exist_ok=True)
            write_csv(pd.DataFrame({'原region索引':selected,'原region':train.var_names[selected],
                                    'ANOVA':scores[selected],'填补中位数':fill[selected]}),
                      reference/'A3_selected_regions.csv')
            del train
            temporary = cached.with_suffix('.partial.npz')
            np.savez(temporary, selected=selected, fill=fill, components=pca.components_, mean=pca.mean_,
                     explained_variance_ratio=pca.explained_variance_ratio_)
            temporary.replace(cached)
        with np.load(cached) as cache:
            selected, fill = cache['selected'],cache['fill']
            pca=PCA(n_components=len(cache['components']),svd_solver='randomized',random_state=42)
            pca.components_=cache['components']
            pca.mean_=cache['mean']
            pca.explained_variance_ratio_=cache['explained_variance_ratio']
            pca.n_features_in_=len(selected)
        real_rep=pca.transform(imputed_features(real,real_rows,selected,fill))
        candidate_rep=pca.transform(imputed_features(candidate,rows,selected,fill))
        a3,cells,types=milisi(real_rep,read_labels(real,real_rows),candidate_rep,read_labels(candidate,rows),30)
        write_csv(cells,destination/'A3_cells.csv')
        write_csv(types,destination/'A3_type_details.csv')
        if case=='liu2023/full':
            true_a2=pd.concat([pairs,paired_pearson(
                dense_block(real,real_rows,pairs['原region索引'].to_numpy()),
                dense_block(real,real_rows,pairs['复制region索引'].to_numpy()))],axis=1)
            true_a2.insert(0,'方法','真实数据')
            write_csv(true_a2,destination/'A2_real_duplicate_correlations.csv')
            import umap
            xy=umap.UMAP(n_neighbors=30,random_state=2022,transform_seed=2022,n_jobs=1).fit_transform(np.vstack([real_rep,candidate_rep]))
            coordinates=pd.DataFrame({'方法':method,'来源':np.repeat(['真实','合成'],[real.n_obs,candidate.n_obs]),
                '细胞ID':np.r_[real.obs_names,candidate.obs_names],'UMAP1':xy[:,0],'UMAP2':xy[:,1]})
            write_csv(coordinates,destination/'A3_joint_coordinates.csv')
            (destination/'figures').mkdir(exist_ok=True)
            regions=load_plot_regions(real,natural,PROJECT/'evaluate/A1/marker_regions_liu2023.csv','mm10')
            plot_a1({'真实数据':real,method:candidate},{'真实数据':real_rows,method:rows},[method],regions,destination)
        exploratory = method == 'simscDNAm' and (candidate_path.parents[1]/'exploratory.json').is_file()
        method_name = f'{method}（探索性）' if exploratory else method
        row={'子数据集':case.split('/')[1],'方法':method_name,COLUMNS[0]:a1['加权 1-WD'],COLUMNS[1]:a1['加权 MSE'],
             COLUMNS[2]:a1['加权 Pearson'],COLUMNS[3]:a2['Pearson'].mean(),COLUMNS[4]:a3}
        write_csv(pd.DataFrame([row]),destination/'scores.csv')
    print('评分完成',case,method,flush=True)

def publish():
    columns = ['子数据集', '方法', *COLUMNS]
    all_tables = {}
    for case in CASES:
        scores = [RUN/case/method/'scores.csv' for method in METHODS]
        rows = [read_csv(path) for path in scores if path.is_file()]
        table = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=columns)
        all_tables[case] = table
        study, accession = case.split('/')
        title = f"{study} {'全数据集' if accession == 'full' else accession}"
        write_csv(table.drop(columns='子数据集'), ROOT/'逐数据集表'/f'{title}.csv')

    full = all_tables['liu2023/full']
    full_path = ROOT/'tables/liu2023全数据集.csv'
    plotted_path = ROOT/'internal/evaluation/liu2023/full/plotted.json'
    previous = json.loads(plotted_path.read_text()) if plotted_path.is_file() else []
    write_csv(full.drop(columns='子数据集'), full_path)
    for study, accessions in SUBSETS.items():
        frames = [all_tables[f'{study}/{gse}'] for gse in accessions if len(all_tables[f'{study}/{gse}'])]
        write_csv(pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns),
                  ROOT/'tables'/f'{study}三个子数据集.csv')
    methods = full['方法'].tolist()
    if methods and methods != previous:
        source = plotted_path.parent
        source.mkdir(parents=True, exist_ok=True)
        dirs = [RUN/'liu2023/full'/method for method in methods]
        for name in ['A1_plot_histograms.csv','A1_type_histograms.csv','A2_duplicate_correlations.csv','A3_joint_coordinates.csv']:
            frames = [read_csv(directory/name) for directory in dirs]
            rows = [part.loc[part['方法'].eq(method)] for part, method in zip(frames,methods)]
            if name.startswith('A1_'):
                rows.insert(0,frames[0].loc[frames[0]['方法'].eq('真实数据')])
            if name.startswith('A2_'):
                rows.insert(0,read_csv(dirs[0]/'A2_real_duplicate_correlations.csv'))
            write_csv(pd.concat(rows,ignore_index=True),source/name)
        shutil.copyfile(dirs[0]/'A1_plot_regions.csv', source/'A1_plot_regions.csv')
        (ROOT/'figures').mkdir(exist_ok=True)
        notebook_function('比较图重绘.ipynb','redraw_comparison')(ROOT)
        plotted_path.write_text(json.dumps(methods), encoding='utf-8')

    ablation = ROOT/'ablation/liu2023/full'
    ablation_csv = ablation/'scores.csv'
    public_ablation = ROOT/'tables/liu2023全量消融.csv'
    if ablation_csv.is_file() and (ablation/'visual_verification.json').is_file():
        shutil.copyfile(ablation_csv,public_ablation)
    elif not public_ablation.is_file():
        write_csv(pd.DataFrame(columns=['方法',*COLUMNS]),public_ablation)
    reports = [json.loads(path.read_text(encoding='utf-8')) for case in CASES
               if (path := ROOT/'preprocessing'/case/'filter.json').is_file()]
    (ROOT/'Subclass过滤汇总.json').write_text(json.dumps(reports,ensure_ascii=False,indent=2),encoding='utf-8')
    finished = notebook_function('更新结果README.ipynb','update_results_readme')(ROOT,FIGURE_PREFIX)
    scored = sum(len(table) for table in all_tables.values())
    figures = len(list((ROOT/'figures').glob('*.png')))
    ablated = len(read_csv(public_ablation))
    report = {'状态':'完成' if scored == len(CASES) * len(METHODS) and figures == 6 and ablated == 3 else '过滤后重跑中',
              '训练最少细胞数':10,'过滤完成配置数':len(reports),'主实验表':10,'主实验已评分组合':scored,
              '已评分新增组合':finished,'消融表':1,'消融已评分版本':ablated,'图像数':figures}
    (ROOT/'最终交付核验.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    print('README 已刷新，主实验已评分',scored,f'/{len(CASES) * len(METHODS)}，图',figures,'/6',flush=True)
    return report


In [ ]:
publish()
for case in CASES:
    for method in METHODS:
        record=RUN/f"{case.replace('/','__')}__{method}.json"
        if record.is_file() and json.loads(record.read_text()).get('状态')=='完成':
            score(case, method)
            publish()